In [2]:
%pip install lolalytics-api

Note: you may need to restart the kernel to use updated packages.


In [4]:
from lolalytics_api import matchup

result = matchup(
    champion1="riven",
    champion2="malphite",
    lane="top",
    rank="emerald"
)

print(result)

{
    "winrate": "46.44",
    "number_of_games": "267"
}


In [5]:

tests = [
    ("riven", "malphite"),
    ("riven", "garen"),
    ("riven", "fiora"),
    ("riven", "camille"),
    ("riven", "irelia"),
]

for champ, opponent in tests:
    result = matchup(
        champion1=champ,
        champion2=opponent,
        lane="top",
        rank="emerald"
    )

    print(f"{champ.title()} vs {opponent.title()}: {result}")

Riven vs Malphite: {
    "winrate": "46.44",
    "number_of_games": "267"
}
Riven vs Garen: {
    "winrate": "45.67",
    "number_of_games": "289"
}
Riven vs Fiora: {
    "winrate": "44.57",
    "number_of_games": "184"
}
Riven vs Camille: {
    "winrate": "51.92",
    "number_of_games": "156"
}
Riven vs Irelia: {
    "winrate": "55.73",
    "number_of_games": "131"
}


In [1]:
import json
import pandas as pd
from pathlib import Path
from lolalytics_api import matchup

PROCESSED_DIR = Path("data/processed")

missing = pd.read_csv(
    PROCESSED_DIR / "missing_matchups_16_19.csv"
)

expected = pd.read_csv(
    PROCESSED_DIR / "expected_winrates_16_19.csv"
)

new_rows = []

for _, row in missing.iterrows():

    try:
        result = matchup(
            champion1=str(row["champion"]).lower(),
            champion2=str(row["opponent_champion"]).lower(),
            lane=str(row["lane"]).lower(),
            rank="emerald"
        )

        if isinstance(result, str):
            result = json.loads(result)

        new_rows.append({
            "patch": "16.19",
            "rank": "emerald",
            "lane": row["lane"],
            "champion": row["champion"],
            "opponent_champion": row["opponent_champion"],
            "expected_winrate": float(result["winrate"]),
            "matchup_games": int(
                str(result["number_of_games"]).replace(",", "")
            ),
            "status": "ok"
        })

    except Exception as e:
        print(
            row["champion"],
            "vs",
            row["opponent_champion"],
            "FAILED:",
            e
        )

new_results = pd.DataFrame(new_rows)

expected = pd.concat(
    [expected, new_results],
    ignore_index=True
).drop_duplicates(
    subset=[
        "lane",
        "champion",
        "opponent_champion"
    ],
    keep="last"
)

expected.to_csv(
    PROCESSED_DIR / "expected_winrates_16_19.csv",
    index=False
)

expected.to_parquet(
    PROCESSED_DIR / "expected_winrates_16_19.parquet",
    index=False
)

new_results

Udyr vs Maokai FAILED: list index out of range
Taliyah vs Nasus FAILED: list index out of range


""
